# GRU Model Univariate


In this section we implement the GRU Model (Gated Recurrent Unit) using the **TimeSeriesDataset** approach with one-hot encoding.

The GRU (Gated Recurrent Unit) Forecaster is a gated recurrent neural network designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. The model balances the complexity of LSTMs with the simplicity of vanilla RNNs, using two gates (reset and update) to control information flow.

With this appraoch each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific patterns with fewer parameters than LSTM while still capturing long-term dependencies.

**Layer Breakdown**
- GRU Layers: 2 stacked GRU layers with gating mechanisms
- Hidden Size: 128 units per layer (default)
- Dropout: Applied between GRU layers (if >1 layer) and before final output
- Output Layer: Single fully connected layer producing 1-step forecast

**Advantages**

- Efficient Memory: Fewer parameters than LSTM (~25% reduction) - no separate cell state
- Better than RNN: Handles long-term dependencies better than vanilla RNN through gating
- Faster Training: Simpler gating mechanism (2 gates vs LSTM's 3) leads to faster computation
- Less Overfitting: Fewer parameters reduce overfitting risk on smaller datasets

**Limitations**
- Slightly Less Powerful: May underperform LSTM on very complex patterns requiring detailed memory control
- Still Sequential: Cannot be fully parallelized like CNNs or Transformers
- Gradient Issues: While better than vanilla RNN, can still face gradient problems on very long sequences (>100 steps)

**When to Use GRU**
- Dataset size is limited (prevents LSTM overfitting)
- Sequences are moderately long (10-50 timesteps)
- Pattern complexity is moderate (not requiring LSTM's fine-grained memory control)

## Model

In [ ]:
import torch
import torch.nn as nn

class GRUForecaster(nn.Module):
    """
    GRU model for MULTIVARIATE time series forecasting.
    Architecture: GRU -> Dropout -> GRU -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    
    GRU is similar to LSTM but with fewer parameters (no cell state).
    Generally faster than LSTM while maintaining good performance.
    Uses reset and update gates instead of LSTM's input/forget/output gates.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: GRU hidden dimension
            num_layers: Number of GRU layers
            dropout: Dropout rate
        """
        super(GRUForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # GRU layers
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # GRU forward pass
        # gru_out: (batch_size, seq_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size)
        gru_out, h_n = self.gru(x)
        
        # Take the output from the last time step
        last_output = gru_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

### Model Results with Exogenous Features